# Übung 5: Lineare Optimierung in der Energiewirtschaft

**Ziel:** Lineare Optimierung verstehen: vom Tafelbeispiel per Hand bis zur
Day-Ahead-Optimierung eines Batteriespeichers.

## Lernpfad

1. **Theorie**: allgemeine Form eines linearen Programms (LP)
2. **Beispiel 1**: Cocktail-Bar (Tafelbeispiel, per Hand und mit Linopy)
3. **Beispiel 2**: Kraftwerkseinsatz mit Kohle, Gas und PV (Merit Order, Linopy)
4. **Beispiel 3**: Day-Ahead-Optimierung eines Batteriespeichers mit PyPSA
5. **Übungsaufgaben**: 4. Kraftwerk (Wind) · Heimspeicher mit echten 2025-Daten


In [32]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import linopy
import pypsa


## 5.1 Theorie: Lineare Optimierung

### Allgemeine Form eines linearen Programms (LP)

$$\min_{x \geq 0}\ \underbrace{c^\top x}_{\text{Zielfunktion}}
\qquad \text{u.B.v.} \qquad
\underbrace{A\,x \leq b}_{\text{Nebenbedingungen}}$$

### Bestandteile

| Symbol | Name | Bedeutung |
|--------|------|-----------|
| $x \in \mathbb{R}^n_{\geq 0}$ | **Entscheidungsvektor** | Die $n$ Variablen, die der Optimierer frei wählen darf |
| $c \in \mathbb{R}^n$ | **Kostenvektor** (Zielfunktionskoeffizienten) | Gewichtung jeder Variable in der Zielfunktion; $c_j$ = Kosten pro Einheit $x_j$ |
| $c^\top x$ | **Zielfunktion** (Objektiv) | Skalarprodukt (der Wert, der minimiert wird) |
| $A \in \mathbb{R}^{m \times n}$ | **Restriktionsmatrix** | $A_{ij}$ beschreibt, wie viel Ressource $i$ eine Einheit von $x_j$ verbraucht |
| $b \in \mathbb{R}^m$ | **Rechte-Seite-Vektor** (RHS) | Verfügbare Kapazität/Obergrenze für jede der $m$ Nebenbedingungen |
| $A x \leq b$ | **Nebenbedingungen** | Jede Zeile $i$: $\sum_j A_{ij}\,x_j \leq b_i$; Ressourcenverbrauch $\leq$ Vorrat |

Maximierung wird auf Minimierung zurückgeführt: $\max c^\top x = -\min(-c)^\top x$


## 5.2 Beispiel 1: Cocktail-Bar *(vgl. Tafelbeispiel VL Prof. Zeiselmeier)*

### Aufgabenstellung

Sie verkaufen am Maifest drei Cocktails:

| Cocktail        | Preis    | Weißer Rum | Cointreau | Wodka  | Gin    |
|-----------------|----------|-----------:|----------:|-------:|-------:|
| Daiquiri        | 5,50 €   | 45 ml      | 30 ml     | 0 ml   | 0 ml   |
| Kamikaze        | 4,50 €   | 0 ml       | 30 ml     | 30 ml  | 0 ml   |
| Long Island ITT | 7,00 €   | 20 ml      | 20 ml     | 20 ml  | 20 ml  |

**Vorräte:** 5 l Rum, 6 l Cointreau, 4 l Wodka, 3 l Gin.

### Mathematische Formulierung

Entscheidungsvektor $x \in \mathbb{R}^3_{\geq 0}$ (Anzahl verkaufter Cocktails je Sorte).
Zielfunktion $z = c^\top x$ maximieren:

$$\max\ z = c^\top x \qquad \text{u.B.v.} \qquad A\,x \leq b,\quad x \geq 0$$

$$c = \begin{bmatrix} 5{,}5 \\ 4{,}5 \\ 7 \end{bmatrix},\quad
A = \begin{bmatrix} 45 & 0 & 20 \\ 30 & 30 & 20 \\ 0 & 30 & 20 \\ 0 & 0 & 20 \end{bmatrix},\quad
b = \begin{bmatrix} 5000 \\ 6000 \\ 4000 \\ 3000 \end{bmatrix}$$

### Lösung per Hand

**Schritt 1 — Gin ist exklusiv für Long Island:** $20\,x_3 \leq 3000 \Rightarrow x_3^\ast = 150$.

**Schritt 2 — Restproblem nach Abzug der Long-Island-Anteile:**
Wodka-Bedingung wird zu $30\,x_2 + 20\cdot 150 \leq 4000 \Rightarrow x_2^\ast \leq 33{,}\overline{3}$.
Rum-Bedingung wird zu $45\,x_1 + 20\cdot 150 \leq 5000 \Rightarrow x_1^\ast \leq 44{,}\overline{4}$.
Da der Preis pro Cocktail positiv ist, lohnt sich jeder zusätzlich verkaufte Cocktail — also Maximum nehmen.

**Schritt 3 — Kontrolle Cointreau:**
$30 \cdot 44{,}\overline{4} + 30 \cdot 33{,}\overline{3} + 20 \cdot 150 = 1333{,}3 + 1000 + 3000 = 5333{,}3$ ml.
Von 6000 ml verfügbar, also bleibt ein Rest von ca. 666 ml.

**Schritt 4 — Optimum:**

$$x^\ast = \begin{bmatrix} 44{,}\overline{4} \\ 33{,}\overline{3} \\ 150 \end{bmatrix},\quad
A\,x^\ast = \begin{bmatrix} 5000 \\ 5333{,}3 \\ 4000 \\ 3000 \end{bmatrix},\quad
b - A\,x^\ast = \begin{bmatrix} 0 \\ 666{,}7 \\ 0 \\ 0 \end{bmatrix}
\quad\Rightarrow\quad z^\ast = c^\top x^\ast \approx 1444{,}44\ €$$

Rum, Wodka und Gin sind **vollständig aufgebraucht** ($b - Ax^* = 0$). Cointreau ist die einzige Zutat mit Restmenge.

### Verbindung: Lineares Gleichungssystem (Gaußsche Elimination)

Was haben wir gerade gemacht? Wir wissen aus der Hand-Lösung, dass **drei der vier Vorräte exakt aufgebraucht** werden (Rum, Wodka, Gin). Diese drei Bedingungen liefern drei **Gleichungen** für drei Unbekannte — ein klassisches **LGS**, wie in Klasse 10 oder der Mathevorlesung:

$$A_B\,x^\ast = b_B
\qquad\text{mit}\qquad
A_B = \begin{pmatrix} 0 & 0 & 20 \\ 0 & 30 & 20 \\ 45 & 0 & 20 \end{pmatrix},\quad
b_B = \begin{pmatrix} 3000 \\ 4000 \\ 5000 \end{pmatrix}
\quad\text{(Zeilen: Gin, Wodka, Rum)}$$

Dieses System hat bereits **Dreiecksform** — reines Rückwärtseinsetzen (Gaußsche Elimination ohne Umformungsschritte):

$$\begin{align}
\text{I:}   \quad 0\cdot x_1 + 0\cdot x_2 + 20\cdot x_3  &= 3000 \\
\text{II:}  \quad 0\cdot x_1 + 30\cdot x_2 + 20\cdot x_3 &= 4000 \\
\text{III:} \quad 45\cdot x_1 + 0\cdot x_2 + 20\cdot x_3 &= 5000
\end{align}$$

Rückwärtseinsetzen (Zeile I hat nur $x_3$ → lösen, Ergebnis in II einsetzen → lösen, dann in III):

$$\begin{align}
\text{I:}   \quad 20\,x_3               &= 3000 \quad\Rightarrow\quad x_3^\ast = 150\\
\text{II:}  \quad 30\,x_2 + 20\cdot 150 &= 4000 \quad\Rightarrow\quad x_2^\ast = \tfrac{100}{3} \approx 33{,}\overline{3}\\
\text{III:} \quad 45\,x_1 + 20\cdot 150 &= 5000 \quad\Rightarrow\quad x_1^\ast = \tfrac{400}{9} \approx 44{,}\overline{4}
\end{align}$$

Das LP reduziert sich also auf ein LGS, sobald bekannt ist, welche Vorräte aufgebraucht werden. Der **Simplex-Algorithmus** macht genau das automatisch: er probiert systematisch verschiedene Auswahlen aus und findet diejenige, bei der das resultierende LGS die optimale Lösung liefert.


### In Linopy

In [33]:
# Indexmenge: x_1 = Daiquiri, x_2 = Kamikaze, x_3 = Long Island ITT
cocktails = ["Daiquiri", "Kamikaze", "Long Island"]

# Preisvektor  c ∈ ℝ³   (c_j = Verkaufspreis pro Cocktail j in €)
c = pd.Series([5.5, 4.5, 7.0], index=cocktails)

# Restriktionsmatrix  A ∈ ℝ^{4×3}  (A[i,j] = ml Spirituose i pro Cocktail j)
A = pd.DataFrame(
    {"Daiquiri":    [45, 30,  0,  0],
     "Kamikaze":    [ 0, 30, 30,  0],
     "Long Island": [20, 20, 20, 20]},
    index=["Weißer Rum", "Cointreau", "Wodka", "Gin"],
)

# Rechte-Seite-Vektor  b ∈ ℝ⁴   (b_i = verfügbarer Vorrat Spirituose i in ml)
b = pd.Series({"Weißer Rum": 5000, "Cointreau": 6000, "Wodka": 4000, "Gin": 3000})

m = linopy.Model()

# lower=0 setzt die Nichtnegativität  x_j ≥ 0
# Die Obergrenzen kommen ausschließlich über A x ≤ b ins Modell
x = m.add_variables(lower=0, coords=[cocktails], dims=["cocktail"], name="x")

# Zielfunktion: Linopy kann direkt maximieren — kein Negieren nötig
m.add_objective((c * x).sum(), sense="max")

# Nebenbedingungen  A x ≤ b  (eine Zeile pro Spirituose)
for s in A.index:
    m.add_constraints((A.loc[s] * x).sum() <= b[s], name=s)

m.solve(solver_name="highs", log_to_console=False)

print(f"Maximaler Gewinn: {m.objective.value:.2f} EUR\n")
print(x.solution.to_pandas().round(2))


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 3 primals, 4 duals
Objective: 1.44e+03
Solver model: available
Solver message: Optimal



Maximaler Gewinn: 1444.44 EUR

cocktail
Daiquiri        44.44
Kamikaze        33.33
Long Island    150.00
Name: solution, dtype: float64


In [34]:
# Tatsächlicher Verbrauch:  A x*
verbrauch = A @ x.solution.to_pandas()
pd.DataFrame({"Vorrat b (ml)":              b,
              "Verbraucht Ax* (ml)":        verbrauch.round(1),
              "Rest b−Ax* (ml)":            (b - verbrauch).round(1),
              "Vorrat aufgebraucht?":       (b - verbrauch).round(1) == 0})


,Vorrat b (ml),Verbraucht Ax* (ml),Rest b−Ax* (ml),Vorrat aufgebraucht?
Weißer Rum,5000,5000.0,0.0,True
Cointreau,6000,5333.3,666.7,False
Wodka,4000,4000.0,0.0,True
Gin,3000,3000.0,0.0,True


## 5.3 Beispiel 2: Kraftwerkseinsatz (Merit Order)

### Aufgabenstellung

Ein Stadtwerk deckt den Bedarf über vier Stunden mit drei Kraftwerken zu minimalen Kosten.
Indexmengen: $T = \{1,2,3,4\}$ (Stunden), $I = \{\text{PV},\text{Kohle},\text{Gas}\}$ (Kraftwerke).

| Kraftwerk | Grenzkosten $c_i$ (€/MWh) | Max. Leistung $\bar p_{t,i}$ (MW) |
|-----------|---------------------------:|----------------------------------|
| PV        |  0                         | wetterabhängig (s. u.)           |
| Kohle     | 30                         | 80                               |
| Gas       | 60                         | 100                              |

Bedarf $d_t$ und verfügbare PV-Leistung je Stunde:

| $t$ | $d_t$ (MW) | $\bar p_{t,\text{PV}}$ (MW) |
|----:|-----------:|----------------------------:|
| 1   | 120        |  0                          |
| 2   |  90        | 30                          |
| 3   | 150        | 60                          |
| 4   | 110        | 20                          |

### Allgemeines Problem

Es ist wieder ein lineares Programm — gleiche Struktur wie Beispiel 1:

$$\min_{x \,\geq\, 0}\ c^\top x
\qquad\text{u.B.v.}\qquad A\,x \leq b$$

zusätzlich kommen Gleichungs-Nebenbedingungen vor (für die Bedarfsdeckung — kein Rest möglich, sonst fehlt Strom).

### Konkrete Formulierung

**Entscheidungsvektor** $P \in \mathbb{R}^{|T| \times |I|}_{\geq 0}$: Einspeiseleistung $p_{t,i} \geq 0$ [MW] je Stunde $t$ und Kraftwerk $i$.

$$\min_{P \,\geq\, 0}\; \sum_{t \in T}\sum_{i \in I} c_i\, p_{t,i}
\qquad\text{u.B.v.}\qquad
\begin{cases}
\displaystyle\sum_{i \in I} p_{t,i} = d_t & \forall\, t \in T \quad \text{(Nachfragedeckung)}\\[6pt]
p_{t,i} \leq \bar{p}_{t,i} & \forall\, t \in T,\; i \in I \quad \text{(Kapazitätsgrenzen)}
\end{cases}$$

$$c = \begin{bmatrix} 0 \\ 30 \\ 60 \end{bmatrix},\qquad
d = \begin{bmatrix} 120 \\ 90 \\ 150 \\ 110 \end{bmatrix},\qquad
\bar P = \begin{bmatrix} 0 & 80 & 100 \\ 30 & 80 & 100 \\ 60 & 80 & 100 \\ 20 & 80 & 100 \end{bmatrix}
\quad\small\text{(Zeilen: } t=1\ldots4\text{; Spalten: PV, Kohle, Gas)}$$

### Lösung per Hand: Merit Order

In jeder Stunde wird nach aufsteigenden Grenzkosten eingesetzt: PV (0 €) → Kohle (30 €) → Gas (60 €).

| $t$ | $d_t$ | PV  | Kohle | Gas | freie Gas-Kapazität (MW) | Kosten (€) |
|----:|------:|----:|------:|----:|-------------------------:|-----------:|
| 1   | 120   |   0 |    80 |  40 |  60                      | 4 800 |
| 2   |  90   |  30 |    60 |   0 | 100                      | 1 800 |
| 3   | 150   |  60 |    80 |  10 |  90                      | 3 000 |
| 4   | 110   |  20 |    80 |  10 |  90                      | 3 000 |
| **Σ** |     |     |       |     |                          | **12 600** |

Kohle läuft in jeder Stunde auf Maximalleistung — vollständig ausgelastet. Gas hat freie Kapazität — wird nur soweit nötig eingesetzt.

### Unterschiede zu Beispiel 1

| | Beispiel 1 (Cocktail) | Beispiel 2 (Kraftwerke) |
|---|---|---|
| **Variable** | $x_j$ — Anzahl Cocktails | $p_{t,i}$ — Leistung pro Stunde und Kraftwerk |
| **Index** | 1-D (Sorte $j$) | 2-D (Stunde $t$ × Kraftwerk $i$) |
| **Zielfunktion** | $\max\ c^\top x$ (Gewinn) | $\min \sum_{t,i} c_i\,p_{t,i}$ (Kosten) |
| **Kapazitäts-NB** | $A x \leq b$ (Vorrat) | $p_{t,i} \leq \bar p_{t,i}$ (Leistung) |
| **Deckungs-NB** | — | $\sum_i p_{t,i} = d_t$ als **Gleichung** |

Pro Stunde wird ein eigenes Gleichgewicht aufgestellt; die Stunden sind nur über den gemeinsamen Kostenvektor $c$ gekoppelt (Merit Order gilt zu jeder Stunde gleich).


### In Linopy

In [35]:
# Indexmengen
T = [1, 2, 3, 4]          # T: Zeitschritte
I = ["PV", "Kohle", "Gas"] # I: Kraftwerke

# Grenzkostenvektor  c ∈ ℝ^|I|
c = pd.Series({"PV": 0, "Kohle": 30, "Gas": 60})
c.index.name = "i"         # Linopy braucht den Indexnamen zur Dim-Zuordnung bei p*c

# Bedarfsvektor  d ∈ ℝ^|T|
d = pd.Series([120, 90, 150, 110], index=T)

# Obere Schranken  P̄ ∈ ℝ^{|T|×|I|}
P_max = pd.DataFrame(
    {"PV":    [ 0, 30, 60, 20],
     "Kohle": [80, 80, 80, 80],
     "Gas":   [100,100,100,100]}, index=T,
)

m = linopy.Model()

# p_{t,i} ≥ 0, obere Schranke P̄_{t,i}
p = m.add_variables(lower=0, upper=P_max, coords=[T, I], dims=["t", "i"], name="p")

# min  Σ_{t,i} c_i · p_{t,i}  =  1_T^T P c
m.add_objective((p * c).sum())

# Σ_i p_{t,i} = d_t  ∀ t  (Nachfragedeckung)
for t in T:
    m.add_constraints(p.sel(t=t).sum() == d[t], name=f"d_t{t}")

m.solve(solver_name="highs", log_to_console=False)

dispatch = p.solution.to_pandas().round(1)
print(f"Gesamtkosten: {m.objective.value:,.0f} EUR\n")
print(dispatch)


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 12 primals, 4 duals
Objective: 1.26e+04
Solver model: available
Solver message: Optimal



Gesamtkosten: 12,600 EUR

i    PV  Kohle   Gas
t                   
1   0.0   80.0  40.0
2  30.0   60.0   0.0
3  60.0   80.0  10.0
4  20.0   80.0  10.0


In [36]:
farben = {"PV": "#FFC107", "Kohle": "#5D4037", "Gas": "#FF7043"}

fig = go.Figure()
for kw in dispatch.columns:
    fig.add_bar(name=kw, x=dispatch.index, y=dispatch[kw], marker_color=farben[kw])
fig.add_scatter(x=T, y=d, name="Bedarf",
                mode="lines+markers", line=dict(color="black", dash="dash"))
fig.update_layout(barmode="stack", template="plotly_white",
                  title="Optimaler Kraftwerkseinsatz (Merit Order mit PV)",
                  xaxis_title="Stunde t", yaxis_title="Leistung (MW)",
                  height=380, margin=dict(t=50, b=40, l=50, r=20))
fig.show()


## 5.4 Beispiel 3: Day-Ahead-Optimierung eines Batteriespeichers (PyPSA)

[PyPSA](https://pypsa.readthedocs.io/) ist ein Framework für
Energiesystem-Optimierung. Das LP wird intern mit Linopy aufgebaut; wir
beschreiben das Modell auf der **Komponentenebene** (Bus, Generator, Last,
Speicher).

```
   Day-Ahead-Markt                Bus „electricity"        Batterie
   ┌─────────────┐  kaufen (p>0) ┌──────────────┐  laden  ┌──────────┐
   │  markt      │ ────────────► │              │ ──────► │          │
   │ (bidirekt., │               │              │  entl.  │ Storage  │
   │  marginal_  │ ◄──────────── │              │ ◄────── │          │
   │  cost=price)│ verkauf (p<0) └──────────────┘         └──────────┘
   └─────────────┘
```

Day-Ahead-Preise: synthetische 2024-Daten aus Übung 4 (`strompreise_2024.csv`),
Demo-Ausschnitt 14 Tage.

In [37]:
df_da = pd.read_csv("../3_Zeitreihen/strompreise_2024.csv",
                    index_col=0, parse_dates=True).rename(columns={"price_eur_mwh": "price"})
df_da = df_da.loc["2024-06-01":"2024-06-14"]

bat = {"kapazitaet_mwh": 10.0, "p_max_mw": 5.0, "wirkungsgrad": 0.95, "soc0": 0.5}

n = pypsa.Network()
n.set_snapshots(df_da.index)
n.add("Bus", "electricity")

# Bidirektionaler Markt-Generator: p_min_pu=-1 erlaubt Verkauf, p_max_pu=1 Kauf.
# Bei marginal_cost=price minimiert das LP gleichzeitig Kaufkosten und maximiert
# Verkaufserlöse (negative Kosten = Erlös).
n.add("Generator", "markt", bus="electricity",
      p_nom=1e6,
      p_min_pu=-1.0, p_max_pu=1.0,
      marginal_cost=df_da["price"])

n.add("StorageUnit", "battery", bus="electricity",
      p_nom=bat["p_max_mw"],
      max_hours=bat["kapazitaet_mwh"] / bat["p_max_mw"],
      efficiency_store=bat["wirkungsgrad"], efficiency_dispatch=bat["wirkungsgrad"],
      state_of_charge_initial=bat["soc0"] * bat["kapazitaet_mwh"],
      cyclic_state_of_charge=False)
n.optimize(solver_name="highs", log_to_console=False)

# Marktleistung: positiv = Kauf, negativ = Verkauf (Konvention oben)
p_markt = n.generators_t.p["markt"]
kauf    = p_markt.clip(lower=0)
verkauf = (-p_markt).clip(lower=0)

# Batterie-Leistung: positiv = entladen (in Bus), negativ = laden (aus Bus)
p_bat  = n.storage_units_t.p["battery"]
laden  = (-p_bat).clip(lower=0)
entlad = p_bat.clip(lower=0)
soc    = n.storage_units_t.state_of_charge["battery"]

ladekosten     = (kauf  * df_da["price"]).sum()
entladeerloese = (verkauf * df_da["price"]).sum()
print(pd.Series({
    "Nettogewinn [€]":   entladeerloese - ladekosten,
    "Geladen [MWh]":     laden.sum(),
    "Entladen [MWh]":    entlad.sum(),
    "Ø Kaufpreis":       ladekosten / kauf.sum() if kauf.sum() > 0 else 0,
    "Ø Verkaufspreis":   entladeerloese / verkauf.sum() if verkauf.sum() > 0 else 0,
}).round(2).to_string())


/tmp/ipykernel_2412829/4121106520.py:25: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['electricity'], dtype='str', name='name')


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 1344 primals, 3360 duals
Objective: -1.16e+04
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


Nettogewinn [€]    11640.70
Geladen [MWh]        461.65
Entladen [MWh]       421.39
Ø Kaufpreis           61.24
Ø Verkaufspreis       94.71


In [38]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=("Day-Ahead-Preis", "Lade-/Entladeleistung", "SOC"))

fig.add_scatter(x=df_da.index, y=df_da["price"], name="Preis",
                line=dict(color="#E64A19"), row=1, col=1)
fig.add_bar(x=df_da.index, y=entlad,  name="Entladen", marker_color="#43A047", row=2, col=1)
fig.add_bar(x=df_da.index, y=-laden,  name="Laden",    marker_color="#1E88E5", row=2, col=1)
fig.add_scatter(x=df_da.index, y=soc,  name="SOC",
                line=dict(color="#8E24AA"), fill="tozeroy", row=3, col=1)

fig.update_yaxes(title_text="€/MWh", row=1, col=1)
fig.update_yaxes(title_text="MW",    row=2, col=1)
fig.update_yaxes(title_text="MWh",   row=3, col=1)
fig.update_layout(template="plotly_white", height=620, hovermode="x unified",
                  barmode="overlay", margin=dict(t=50, b=40, l=50, r=20),
                  title="Batterie-Day-Ahead-Optimierung (synthet. 2024-Preise, 14 Tage)")
fig.show()

## 5.5 Übungsaufgaben

### Aufgabe 1: Viertes Kraftwerk im 4-Stunden-Beispiel

Erweitere das Modell aus **Beispiel 2** um ein viertes Kraftwerk:

| Kraftwerk | Grenzkosten (€/MWh) | Max. Leistung (MW) | Verfügbarkeit |
|-----------|--------------------:|-------------------:|---------------|
| **Wind**  | 0                   | 60                 | $\bar p^{\text{Wind}} = (40, 20, 10, 50)$ MW |

**Aufgaben:**

1. Erweitere das Linopy-Modell um Wind (variable Verfügbarkeit pro Stunde).
2. Berechne die neuen Gesamtkosten und vergleiche mit Beispiel 2 (12 600 €).
3. Plotte den Einsatzplan analog zur Vorlage als gestapeltes Säulendiagramm.
4. **Interpretation:** Welche Kraftwerke sind in welchen Stunden im Einsatz?
   Warum verdrängt Wind ausgerechnet Gas und nicht Kohle?

> *Hinweis:* Wind hat $c_\text{Wind} = 0$, also gleiche Grenzkosten wie PV; die
> Merit-Order setzt zuerst PV + Wind ein, dann Kohle, dann Gas.


In [39]:
# Indexmengen (Wind ergänzt)
T = [1, 2, 3, 4]
I = ["PV", "Wind", "Kohle", "Gas"]

# Grenzkostenvektor  c ∈ ℝ^I
c = pd.Series({"PV": 0, "Wind": 0, "Kohle": 30, "Gas": 60})
c.index.name = "i"         # Linopy braucht den Indexnamen zur Dim-Zuordnung bei p*c

# Bedarfsvektor  d ∈ ℝ^T
d = pd.Series([120, 90, 150, 110], index=T)

# Obere Schranken  P̄ = (p̄_{t,i}) ∈ ℝ^{T×I}
P_max = pd.DataFrame(
    {"PV":    [ 0, 30, 60, 20],
     "Wind":  [40, 20, 10, 50],
     "Kohle": [80, 80, 80, 80],
     "Gas":   [100,100,100,100]}, index=T,
)

m = linopy.Model()
p = m.add_variables(lower=0, upper=P_max,
                    coords=[T, I], dims=["t", "i"], name="p")
m.add_objective((p * c).sum())                              # min 1ᵀ P c
for t in T:
    m.add_constraints(p.sel(t=t).sum() == d[t], name=f"d_t{t}")
m.solve(solver_name="highs", log_to_console=False)

dispatch = p.solution.to_pandas().round(1)
print(f"Gesamtkosten (mit Wind): {m.objective.value:,.0f} EUR")
print(f"Einsparung vs. Beispiel 2: {12600 - m.objective.value:,.0f} EUR\n")
print(dispatch)


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 16 primals, 4 duals
Objective: 7.20e+03
Solver model: available
Solver message: Optimal



Gesamtkosten (mit Wind): 7,200 EUR
Einsparung vs. Beispiel 2: 5,400 EUR

i    PV  Wind  Kohle  Gas
t                        
1   0.0  40.0   80.0 -0.0
2  30.0  20.0   40.0  0.0
3  60.0  10.0   80.0 -0.0
4  20.0  50.0   40.0  0.0


In [40]:
farben = {"PV": "#FFC107", "Wind": "#64B5F6", "Kohle": "#5D4037", "Gas": "#FF7043"}

fig = go.Figure()
for kw in dispatch.columns:
    fig.add_bar(name=kw, x=dispatch.index, y=dispatch[kw], marker_color=farben[kw])
fig.add_scatter(x=T, y=d, name="Bedarf",
                mode="lines+markers", line=dict(color="black", dash="dash"))
fig.update_layout(barmode="stack", template="plotly_white",
                  title="Kraftwerkseinsatz mit Wind (Merit Order)",
                  xaxis_title="Stunde t", yaxis_title="Leistung (MW)",
                  height=380, margin=dict(t=50, b=40, l=50, r=20))
fig.show()


### Aufgabe 2: Heimspeicher der Studentin (echte 2025-Daten + Parameter-Variation)

In **Übung 3** habt ihr dynamische vs. statische Stromtarife verglichen und dabei
festgestellt, dass ein reiner Wechsel auf den dynamischen DA-Tarif für die Studentin
**CHR11 kaum Einsparung bringt**, denn ohne Flexibilität lässt sich der Preisvorteil
nicht nutzen.

Die naheliegende Folgefrage: **Lohnt sich ein kleiner Heimspeicher?**
Baut das PyPSA-Modell aus **Beispiel 3** nach, mit zwei Unterschieden:

| | Beispiel 3 | Aufgabe 2 |
|---|---|---|
| Netz | nur Speicher + Markt | Speicher + Haushaltslast (Load) |
| Tarif | reiner DA-Preis (Kauf & Verkauf) | DA + 200 €/MWh Retail-Aufschlag, **keine** Einspeisung |
| Zeitreihe | synthet. 2024-Daten, 1 h | CHR11-Daten 2025, **15 min** |

```
   DA-Markt                   Bus „haus"              Speicher
   ┌─────────────┐  kaufen   ┌───────────┐  laden   ┌──────────┐
   │ netz        │ ────────► │           │ ───────► │          │
   │ (DA+200 €)  │           │           │          │ StorageU │
   └─────────────┘           │           │  entl.   │          │
                             │           │ ◄─────── └──────────┘
                             └─────┬─────┘
                                   │ Last (CHR11)
                             ┌─────▼─────┐
                             │   Load    │
                             └───────────┘
```

**Daten aus Übung 3:**
- `dayahead_2025.csv`: Day-Ahead-Preise (15 min, Spalte `dayahead €/MWh`)
- `CHR11_last_15min_2025.csv`: Lastprofil (15 min, Spalte `last [kW]`)

Die Studentin bezieht Strom zu $p_\text{Haushalt}(t) = p_\text{DA}(t) + 20\ \text{ct/kWh}$
($= 200\ €/\text{MWh}$ Retail-Aufschlag).

**Aufgaben (alle für einen einzelnen Beispieltag, z. B. `2025-06-15`):**

1. Lade beide CSVs ein und merge sie zu einem DataFrame `df` mit **15-min-Index**
   (tz-naiv UTC). Filtere auf einen Beispieltag (z. B. `df.loc["2025-06-15"]`).
2. Schreibe eine Funktion `optimiere_heimspeicher(df_day, kapazitaet_kwh, leistung_kw, eta=0.95, retail_margin=200.0)` analog zu
   Beispiel 3, mit `Load`, `Generator` (nur Netzbezug, kein Verkauf) und
   `StorageUnit`. Nutze `n.set_snapshots(df_day.index, weightings_from_timedelta=True)`,
   damit die 15-min-Schritte automatisch mit Gewicht 0,25 h gerechnet werden.
   Die Funktion gibt das `Network`-Objekt zurück; die Tagesrechnung steht in `n.objective` (EUR/Tag).
3. **Referenz ohne Speicher** berechnen (Netzwerk **ohne** StorageUnit) und mit einem
   Standard-Speicher (5 kWh / 2,5 kW). Vergleiche die beiden Tagesrechnungen — wie groß ist die Einsparung?
4. **Parametervariation:** Rufe `optimiere_heimspeicher` in einer Schleife auf für
   Speichergrößen von $0{,}5,\ 1,\ 2,\ 4,\ 6,\ 8,\ 10$ kWh
   (C-Rate konstant: $p_\text{max} = \text{Kapazität} / 2\,\text{h}$).
   Plotte die Ersparnis in **€/Tag** und in **% der Tagesreferenzkosten** gegen die Kapazität.
5. **Wirtschaftlichkeit:** Nimm an die Speicherkosten betragen 200 €/kWh
   ([Preisvergleich Balkonkraftwerk-Speicher, idealo](https://www.idealo.de/preisvergleich/Liste/120478587/batteriespeicher-fuer-balkonkraftwerk.html)).
   Skaliere die Tagesersparnis aus Schritt 4 auf ein Jahr (× 365). Berechne pro Speichergröße:
   - Anschaffungskosten (200 €/kWh × Kapazität)
   - Jährliche Ersparnis (= Tagesersparnis × 365)
   - Amortisationszeit in Jahren (Anschaffungskosten / jährliche Ersparnis)
   Plotte oder tabelliere das Ergebnis. Ab welcher Speichergröße nimmt der Grenznutzen stark ab?
   Welcher Faktor begrenzt den maximalen Nutzen (Kapazität oder Leistung)?


In [41]:
# Schritt 1: Daten laden (15-min-Auflösung 2025) + DA-Preis in €/kWh umrechnen
df_da_25 = pd.read_csv("../3_Zeitreihen/dayahead_2025.csv",
                       index_col=0, parse_dates=True)["dayahead €/MWh"]
df_load  = pd.read_csv("../3_Zeitreihen/CHR11_last_15min_2025.csv",
                       index_col=0, parse_dates=True)["last [kW]"]

df = pd.concat([df_da_25, df_load], axis=1).dropna()
df.columns = ["price_eur_mwh", "load_kw"]
df.index = df.index.tz_convert("UTC").tz_localize(None)

# Endkunden-freundliche Einheiten:
#   price [€/kWh] = price [€/MWh] / 1000
#   cost_t [€/Slot] = (price + retail_aufschlag) [€/kWh] * load [kW] * 0.25 h
df["price"] = df["price_eur_mwh"] / 1000.0            # €/kWh — Endkunden-Skala
df["load"]  = df["load_kw"]                            # kW — unverändert
RETAIL_EUR_KWH = 0.20                                  # 20 ct/kWh Aufschlag

print(f"Zeitauflösung: {df.index[1] - df.index[0]}")
print(f"Datenpunkte gesamt: {len(df)}")
print(f"DA-Preis-Spannweite: {df['price'].min()*100:.1f} – {df['price'].max()*100:.1f} ct/kWh")
print(f"Retail-Aufschlag: {RETAIL_EUR_KWH*100:.0f} ct/kWh")
df[["price", "load"]].head()


Zeitauflösung: 0 days 00:15:00
Datenpunkte gesamt: 35040
DA-Preis-Spannweite: -50.0 – 83.5 ct/kWh
Retail-Aufschlag: 20 ct/kWh


,price,load
timestamp,,
2024-12-31 23:00:00,0.02240,0.031262
2024-12-31 23:15:00,-0.00740,0.051789
2024-12-31 23:30:00,-0.01016,0.047690
2024-12-31 23:45:00,-0.00530,0.030744
2025-01-01 00:00:00,0.00554,0.031037


In [42]:
# Schritt 1b: Last + Endkundenpreis visualisieren (Jahr + Beispieltag, 4× untereinander)
TAG = "2025-06-15"

# Endkundenpreis = DA-Preis + Retail-Aufschlag (was der Haushalt tatsächlich zahlt)
df_price_ct  = (df["price"] + RETAIL_EUR_KWH) * 100             # ct/kWh, 15-min
kosten_t     = (df["price"] + RETAIL_EUR_KWH) * df["load"] * 0.25  # € pro Slot

# Aggregate Jahr → Tageswerte (Mittel)
load_daily_kw     = df["load"].resample("1D").mean()
price_daily_ctkwh = df_price_ct.resample("1D").mean()

# Beispieltag
tag_df       = df.loc[TAG]
tag_price_ct = df_price_ct.loc[TAG]

# Layout: 4 Zeilen untereinander
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.10,
    subplot_titles=(
        "Last 2025 — Tagesmittel über das Jahr",
        f"Last am {TAG} (96 × 15-min)",
        f"Endkundenpreis 2025 (DA + {RETAIL_EUR_KWH*100:.0f} ct/kWh) — Tagesmittel",
        f"Endkundenpreis am {TAG} — 96 × 15-min",
    ),
)

LAST_COLOR  = "#37474F"
PRICE_COLOR = "#D32F2F"

# Zeile 1 — Last Jahr
fig.add_scatter(x=load_daily_kw.index, y=load_daily_kw.values, mode="lines",
                line=dict(color=LAST_COLOR, width=1),
                showlegend=False, row=1, col=1)
fig.update_yaxes(title_text="kW", row=1, col=1)

# Zeile 2 — Last Beispieltag
fig.add_scatter(x=tag_df.index, y=tag_df["load"].values, mode="lines",
                line=dict(color=LAST_COLOR, width=1.6),
                showlegend=False, row=2, col=1)
fig.update_yaxes(title_text="kW", row=2, col=1)

# Zeile 3 — Endkundenpreis Jahr
fig.add_scatter(x=price_daily_ctkwh.index, y=price_daily_ctkwh.values, mode="lines",
                line=dict(color=PRICE_COLOR, width=1),
                showlegend=False, row=3, col=1)
fig.update_yaxes(title_text="ct/kWh", row=3, col=1)

# Zeile 4 — Endkundenpreis Beispieltag
fig.add_scatter(x=tag_price_ct.index, y=tag_price_ct.values, mode="lines",
                line=dict(color=PRICE_COLOR, width=1.6),
                showlegend=False, row=4, col=1)
fig.update_yaxes(title_text="ct/kWh", row=4, col=1)

fig.update_layout(
    template="plotly_white",
    height=900, margin=dict(t=80, b=40, l=70, r=30),
    title_text="Last und Endkundenpreis (CHR11 2025)",
    title_y=0.985,
)
fig.show()

print(f"Jahressumme Last:                    {(df['load']*0.25).sum():.0f} kWh")
print(f"Jahressumme Stromkosten:             {kosten_t.sum():.0f} €")
print(f"Durchschnitts-Endkundenpreis Jahr:   "
      f"{kosten_t.sum()/(df['load']*0.25).sum()*100:.1f} ct/kWh")
print(f"Endkundenpreis-Spannweite Jahr:      "
      f"{df_price_ct.min():.1f} – {df_price_ct.max():.1f} ct/kWh")
print(f"Endkundenpreis-Spannweite Beispieltag: "
      f"{tag_price_ct.min():.1f} – {tag_price_ct.max():.1f} ct/kWh")


Jahressumme Last:                    887 kWh
Jahressumme Stromkosten:             266 €
Durchschnitts-Endkundenpreis Jahr:   30.0 ct/kWh
Endkundenpreis-Spannweite Jahr:      -30.0 – 103.5 ct/kWh
Endkundenpreis-Spannweite Beispieltag: 17.5 – 35.6 ct/kWh


In [43]:
# Schritt 2: Funktion optimiere_heimspeicher (15-min, €/kWh-Input)
def optimiere_heimspeicher(df_day, kapazitaet_kwh, leistung_kw,
                           eta=0.95, retail_eur_kwh=0.20,
                           ohne_speicher=False):
    """Optimiert einen Heimspeicher für einen Tag mit 15-min-Zeitschritten.

    Parameters
    ----------
    df_day        : DataFrame mit Spalten 'price' [€/kWh] und 'load' [kW]
    kapazitaet_kwh: Speicherkapazität [kWh] (ignoriert wenn ohne_speicher=True)
    leistung_kw   : Lade-/Entladeleistung [kW] (ignoriert wenn ohne_speicher=True)
    eta           : Round-Trip-Wirkungsgrad-Wurzel (sqrt(0.9025) ≈ 0.95)
    retail_eur_kwh: Aufschlag auf den DA-Preis [€/kWh] (Default 0.20 = 20 ct/kWh)
    ohne_speicher : True = nur Netz + Last (Referenz)

    Returns
    -------
    n : pypsa.Network — Tagesrechnung in n.objective [€/Tag]
    """
    n = pypsa.Network()
    n.set_snapshots(df_day.index, weightings_from_timedelta=True)
    n.add("Bus", "haus")

    # Haushaltslast: kW → MW (PyPSA-Konvention)
    n.add("Load", "last", bus="haus",
          p_set=df_day["load"].values / 1e3)

    # Netzbezug: marginal_cost in PyPSA-Konvention €/MWh = €/kWh × 1000
    n.add("Generator", "netz", bus="haus",
          marginal_cost=(df_day["price"].values + retail_eur_kwh) * 1000.0,
          p_nom=1e6)

    if not ohne_speicher:
        n.add("StorageUnit", "speicher", bus="haus",
              p_nom=leistung_kw / 1e3,
              max_hours=kapazitaet_kwh / leistung_kw,
              efficiency_store=eta, efficiency_dispatch=eta,
              cyclic_state_of_charge=True)

    n.optimize(solver_name="highs", log_to_console=False)
    return n


# Beispieltag: 15. Juni 2025
tag = df.loc[TAG]

# Referenz: OHNE Speicher
n_ref = optimiere_heimspeicher(tag, 0, 0, ohne_speicher=True)
kosten_ohne = n_ref.objective

# Standard-Speicher: 5 kWh / 2,5 kW
n_std = optimiere_heimspeicher(tag, kapazitaet_kwh=5.0, leistung_kw=2.5)
kosten_mit = n_std.objective

ersparnis = kosten_ohne - kosten_mit
ersparnis_pct = 100 * ersparnis / kosten_ohne
print(f"Tagesrechnung ohne Speicher:      {kosten_ohne:7.2f} €/Tag")
print(f"Tagesrechnung mit 5 kWh / 2,5 kW: {kosten_mit:7.2f} €/Tag")
print(f"Ersparnis:                        {ersparnis:7.2f} €/Tag = {ersparnis_pct:.1f} %")


/tmp/ipykernel_2412829/1300388208.py:40: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['haus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 96 primals, 288 duals
Objective: 9.56e-01
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper were not assigned to the network.
/tmp/ipykernel_2412829/1300388208.py:40: FutureWarning:

The default value of `include_objective_constant` will change from True to False in ve

Tagesrechnung ohne Speicher:         0.96 €/Tag
Tagesrechnung mit 5 kWh / 2,5 kW:    0.68 €/Tag
Ersparnis:                           0.27 €/Tag = 28.5 %


In [44]:
# Schritt 4: Parametervariation — Speichergröße vs. Ersparnis (€/Tag und €/Jahr)
kapazitaeten = [0.5, 1, 2, 4, 6, 8, 10]  # kWh
DAYS_PER_YEAR = 365

ergebnisse = []
for kap in kapazitaeten:
    n_k = optimiere_heimspeicher(tag, kapazitaet_kwh=kap, leistung_kw=kap / 2)
    einsparung_eur_tag  = kosten_ohne - n_k.objective
    einsparung_eur_jahr = einsparung_eur_tag * DAYS_PER_YEAR
    einsparung_pct      = 100 * einsparung_eur_tag / kosten_ohne
    ergebnisse.append({
        "kapazitaet_kwh":      kap,
        "einsparung_eur_tag":  round(einsparung_eur_tag, 4),
        "einsparung_eur_jahr": round(einsparung_eur_jahr, 1),
        "einsparung_pct":      round(einsparung_pct, 2),
    })

df_studie = pd.DataFrame(ergebnisse).set_index("kapazitaet_kwh")
print(df_studie)

# Eine Grafik mit zwei Y-Achsen: Bars = Ersparnis €/Jahr, Linie = % der Tagesreferenz
fig = go.Figure()
fig.add_bar(x=df_studie.index, y=df_studie["einsparung_eur_jahr"],
            name="Ersparnis (€/Jahr)", marker_color="#43A047",
            text=[f"{v:.0f} €" for v in df_studie["einsparung_eur_jahr"]],
            textposition="outside")
fig.add_scatter(x=df_studie.index, y=df_studie["einsparung_pct"],
                mode="lines+markers", name="Ersparnis (% der Stromkosten)",
                line=dict(color="#1E88E5", width=3), yaxis="y2")
fig.update_layout(
    template="plotly_white",
    title=f"Heimspeicher: Jahresersparnis vs. Speichergröße (Tag {tag.index[0].date()} hochgerechnet)",
    xaxis_title="Speicherkapazität (kWh)",
    yaxis=dict(title="Ersparnis (€/Jahr)", side="left"),
    yaxis2=dict(title="Ersparnis (% der Stromkosten)", overlaying="y", side="right",
                showgrid=False),
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.8)"),
    height=420, margin=dict(t=70, b=50, l=60, r=60),
)
fig.show()


/tmp/ipykernel_2412829/1300388208.py:40: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['haus'], dtype='str', name='name')


INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 384 primals, 960 duals
Objective: 8.88e-01
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.
/tmp/ipykernel_2412829/1300388208.py:40: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by n

                einsparung_eur_tag  einsparung_eur_jahr  einsparung_pct
kapazitaet_kwh                                                         
0.5                         0.0674                 24.6            7.05
1.0                         0.1216                 44.4           12.73
2.0                         0.2195                 80.1           22.96
4.0                         0.2688                 98.1           28.13
6.0                         0.2748                100.3           28.75
8.0                         0.2785                101.7           29.14
10.0                        0.2814                102.7           29.44


In [45]:
# Schritt 5: Wirtschaftlichkeit — Anschaffungskosten vs. Lebensdauer-Ersparnis
COSTS_EUR_PER_KWH = 200.0
LIFETIME_YEARS    = 10        # typische Garantielebensdauer Heimspeicher

df_wirtschaft = df_studie.copy()
df_wirtschaft["anschaffung_eur"]      = df_wirtschaft.index * COSTS_EUR_PER_KWH
df_wirtschaft["ersparnis_10y_eur"]    = df_wirtschaft["einsparung_eur_jahr"] * LIFETIME_YEARS
df_wirtschaft["netto_10y_eur"]        = df_wirtschaft["ersparnis_10y_eur"] - df_wirtschaft["anschaffung_eur"]
df_wirtschaft["amortisation_jahre"]   = (
    df_wirtschaft["anschaffung_eur"] / df_wirtschaft["einsparung_eur_jahr"]
).round(1)

print(df_wirtschaft.round(2))

# Side-by-side Bars: Anschaffung vs Lebensdauer-Ersparnis pro Speichergröße
fig = go.Figure()
fig.add_bar(x=df_wirtschaft.index, y=df_wirtschaft["anschaffung_eur"],
            name=f"Anschaffung ({COSTS_EUR_PER_KWH:.0f} €/kWh)",
            marker_color="#E64A19",
            text=[f"{v:.0f} €" for v in df_wirtschaft["anschaffung_eur"]],
            textposition="outside")
fig.add_bar(x=df_wirtschaft.index, y=df_wirtschaft["ersparnis_10y_eur"],
            name=f"Ersparnis nach {LIFETIME_YEARS} Jahren",
            marker_color="#43A047",
            text=[f"{v:.0f} €" for v in df_wirtschaft["ersparnis_10y_eur"]],
            textposition="outside")
fig.update_layout(
    barmode="group",
    template="plotly_white",
    title=f"Wirtschaftlichkeit: Anschaffung vs. {LIFETIME_YEARS}-Jahres-Ersparnis",
    xaxis_title="Speicherkapazität (kWh)",
    yaxis_title="Euro",
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.8)"),
    height=420, margin=dict(t=70, b=50, l=60, r=20),
)
fig.show()

# Konkrete Aussage als Print-Zusammenfassung
beste_idx = df_wirtschaft["netto_10y_eur"].idxmax()
beste_netto = df_wirtschaft.loc[beste_idx, "netto_10y_eur"]
beste_amo = df_wirtschaft.loc[beste_idx, "amortisation_jahre"]

print()
print("=" * 65)
print(f"  Wirtschaftlichkeits-Zusammenfassung (Tag {tag.index[0].date()})")
print("=" * 65)
for kap in [1, 2, 5, 10]:
    if kap not in df_wirtschaft.index:
        # 5 kWh nicht in Sweep-Index — naechsten Wert nehmen
        if kap == 5: kap = 6
        if kap == 1: kap = 1
        if kap not in df_wirtschaft.index: continue
    row = df_wirtschaft.loc[kap]
    print(f"  {kap:>4.0f} kWh:  Anschaffung {row['anschaffung_eur']:>5.0f} € · "
          f"Ersparnis {row['einsparung_eur_jahr']:>5.1f} €/Jahr · "
          f"Amortisation {row['amortisation_jahre']:>5.1f} Jahre · "
          f"Netto nach 10y: {row['netto_10y_eur']:+.0f} €")
print("=" * 65)
print(f"  Beste Größe (max. 10y-Netto): {beste_idx} kWh "
      f"→ Netto {beste_netto:+.0f} €, Amortisation {beste_amo} Jahre")
print("=" * 65)


                einsparung_eur_tag  einsparung_eur_jahr  einsparung_pct  \
kapazitaet_kwh                                                            
0.5                           0.07                 24.6            7.05   
1.0                           0.12                 44.4           12.73   
2.0                           0.22                 80.1           22.96   
4.0                           0.27                 98.1           28.13   
6.0                           0.27                100.3           28.75   
8.0                           0.28                101.7           29.14   
10.0                          0.28                102.7           29.44   

                anschaffung_eur  ersparnis_10y_eur  netto_10y_eur  \
kapazitaet_kwh                                                      
0.5                       100.0              246.0          146.0   
1.0                       200.0              444.0          244.0   
2.0                       400.0              801


  Wirtschaftlichkeits-Zusammenfassung (Tag 2025-06-15)
     1 kWh:  Anschaffung   200 € · Ersparnis  44.4 €/Jahr · Amortisation   4.5 Jahre · Netto nach 10y: +244 €
     2 kWh:  Anschaffung   400 € · Ersparnis  80.1 €/Jahr · Amortisation   5.0 Jahre · Netto nach 10y: +401 €
     6 kWh:  Anschaffung  1200 € · Ersparnis 100.3 €/Jahr · Amortisation  12.0 Jahre · Netto nach 10y: -197 €
    10 kWh:  Anschaffung  2000 € · Ersparnis 102.7 €/Jahr · Amortisation  19.5 Jahre · Netto nach 10y: -973 €
  Beste Größe (max. 10y-Netto): 2.0 kWh → Netto +401 €, Amortisation 5.0 Jahre


### Schritt 6: Wann/wo wird eingekauft? — Beispieltag vor/nach Optimierung

Statt 4 Beispielwochen zeigt der folgende Plot kompakt **einen einzelnen Tag**
(Standard: `2025-06-15`) und vergleicht **Vorher** (kein Speicher, Netzbezug =
Last) mit **Nachher** (2 kWh / 1 kW Speicher, Netzbezug verschiebt sich auf
billige Stunden). Die zweite y-Achse zeigt den DA-Preis als Treiber.


In [46]:
# Schritt 6: Beispieltag — Netzbezug Vorher vs Nachher (2 kWh / 1 kW)
KAP, LEIS = 2.0, 1.0   # bester Trade-off aus Schritt 5

n_opt = optimiere_heimspeicher(tag, KAP, LEIS)

# Zeitreihen aus PyPSA holen (MW → kW)
netz_vorher_kw  = n_ref.generators_t.p["netz"]      * 1e3   # = Last (kein Speicher)
netz_nachher_kw = n_opt.generators_t.p["netz"]      * 1e3   # Last + Laden − Entladen
p_speicher_kw   = n_opt.storage_units_t.p["speicher"] * 1e3
entladung_kw    = p_speicher_kw.clip(lower=0)
ladung_kw       = (-p_speicher_kw).clip(lower=0)
last_kw         = tag["load"]

# Endkundenpreis = DA + Retail-Aufschlag (in ct/kWh) — was die Studentin zahlt
endkunde_ct_kwh = (tag["price"] + RETAIL_EUR_KWH) * 100

# Farb-Konvention
PRICE_COLOR   = "#D32F2F"
LAST_COLOR    = "#37474F"
NACHHER_COLOR = "#43A047"
LADUNG_FILL   = "rgba(255,167,38,0.30)"
ENTLAD_FILL   = "rgba(67,160,71,0.30)"

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_scatter(x=last_kw.index, y=last_kw.values, mode="lines",
                line=dict(color=LAST_COLOR, width=1.5),
                name="Haushaltslast")
fig.add_scatter(x=netz_vorher_kw.index, y=netz_vorher_kw.values, mode="lines",
                line=dict(color=LAST_COLOR, width=2, dash="dot"),
                name="Netzbezug VORHER (ohne Speicher)")
fig.add_scatter(x=netz_nachher_kw.index, y=netz_nachher_kw.values, mode="lines",
                line=dict(color=NACHHER_COLOR, width=2),
                name="Netzbezug NACHHER (mit Speicher)")
fig.add_scatter(x=ladung_kw.index, y=ladung_kw.values, mode="lines",
                line=dict(width=0),
                fill="tozeroy", fillcolor=LADUNG_FILL,
                name="Speicher lädt (Netz → Akku)", hoverinfo="skip")
fig.add_scatter(x=entladung_kw.index, y=entladung_kw.values, mode="lines",
                line=dict(width=0),
                fill="tozeroy", fillcolor=ENTLAD_FILL,
                name="Speicher entlädt (Akku → Last)", hoverinfo="skip")

# Rechte Achse: ENDKUNDENPREIS (DA + Retail) in ct/kWh, rot
fig.add_scatter(x=tag.index, y=endkunde_ct_kwh.values, mode="lines",
                line=dict(color=PRICE_COLOR, width=1.4, dash="dash"),
                name=f"Endkundenpreis [ct/kWh] (DA + {RETAIL_EUR_KWH*100:.0f})",
                yaxis="y2")

fig.update_yaxes(title_text="Leistung [kW]", secondary_y=False)
fig.update_yaxes(title_text="Endkundenpreis [ct/kWh]", secondary_y=True, color=PRICE_COLOR)
fig.update_layout(
    template="plotly_white",
    title=dict(
        text=f"Beispieltag {TAG}: Wann/wo wird eingekauft? — Speicher 2 kWh / 1 kW",
        y=0.97,
    ),
    height=560,
    margin=dict(t=140, b=60, l=70, r=70),
    legend=dict(
        orientation="h", yanchor="bottom", y=1.02,
        xanchor="left", x=0,
        bgcolor="rgba(255,255,255,0)",
        font=dict(size=11),
    ),
    xaxis=dict(title="Zeit (UTC)"),
)
fig.show()

# Kennzahlen Vorher/Nachher
kosten_vorher  = n_ref.objective
kosten_nachher = n_opt.objective
print()
print(f"  Kosten VORHER (ohne Speicher):   {kosten_vorher:6.3f} €")
print(f"  Kosten NACHHER (2 kWh / 1 kW):   {kosten_nachher:6.3f} €")
print(f"  Ersparnis:                       {kosten_vorher - kosten_nachher:6.3f} €  "
      f"({100*(kosten_vorher - kosten_nachher)/kosten_vorher:+.1f} %)")
print(f"  Geladen (Netz → Akku):           {(ladung_kw*0.25).sum():5.2f} kWh")
print(f"  Entladen (Akku → Last):          {(entladung_kw*0.25).sum():5.2f} kWh")


/tmp/ipykernel_2412829/1300388208.py:40: FutureWarning:

The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.

Index(['haus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - log_to_console: False
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 384 primals, 960 duals
Objective: 7.36e-01
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, Storag


  Kosten VORHER (ohne Speicher):    0.956 €
  Kosten NACHHER (2 kWh / 1 kW):    0.736 €
  Ersparnis:                        0.219 €  (+23.0 %)
  Geladen (Netz → Akku):            2.43 kWh
  Entladen (Akku → Last):           2.20 kWh


### Interpretation Aufgabe 2

**Caveat zur Jahreshochrechnung:** Tagesersparnis × 365 ist nur eine grobe
Obergrenze. Der Beispieltag (15. Juni) liegt im Sommer. Da gibts viel Sonne, niedrigeres
Preisniveau, aber ein **großer Tag-Nacht-Spread** durch PV-Mittagstief + Abendpeak.
An Wintertagen ist der Spread des DA-Preis flacher, der Speicher verdient weniger.

Ausblick: Für eine ehrliche Jahresrechnung und Potentialabschätzung müsste man 365 Tage optimieren. --> Studentische Projektarbeit Thema 6